# Dividend and multiplier timeline research

This notebook compares the dividend lifecycle in `output/nasdaq.csv` with the on-chain multiplier events collected for Robinhood Stock Tokens and xStocks. It studies event timing, checks the collected xStocks timestamps against the provider's documented scheduling rule, and estimates the dividend value implied by each multiplier change for comparison with the Nasdaq declaration.

The comparisons are descriptive: matching dates suggests a timing relationship but does not prove which dividend caused an on-chain event. All provider timestamps are treated as UTC.

In [1]:
from csv import DictReader
from datetime import date, datetime, time, timedelta, timezone
from decimal import Decimal, localcontext
from html import escape
from pathlib import Path
from zoneinfo import ZoneInfo

import requests

try:
    from IPython.display import HTML, Markdown, display
except ImportError:  # Allows the analysis code to run outside Jupyter too.
    HTML = Markdown = lambda value: value
    display = print


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'output' / 'nasdaq.csv').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from the repository or one of its subdirectories.')


def read_csv_records(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return list(DictReader(handle))


def parse_utc(value):
    return datetime.fromisoformat(value.replace('Z', '+00:00')).astimezone(timezone.utc)


def format_value(value):
    if isinstance(value, datetime):
        return value.astimezone(timezone.utc).isoformat().replace('+00:00', 'Z')
    return str(value)


def show_table(rows, columns=None):
    rows = list(rows)
    if not rows:
        display(Markdown('_No rows._'))
        return
    columns = columns or list(rows[0])
    header = ''.join(f'<th>{escape(str(column))}</th>' for column in columns)
    body = ''.join(
        '<tr>' + ''.join(f'<td>{escape(format_value(row.get(column, "")))}</td>' for column in columns) + '</tr>'
        for row in rows
    )
    display(HTML(
        '<div style="overflow-x:auto;max-width:100%">'
        '<table style="border-collapse:collapse;white-space:nowrap">'
        f'<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'
    ))


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / 'output'
UTC = timezone.utc
REPO_ROOT

PosixPath('/Users/robert/Work/dividends-comparison')

## Nasdaq dividend events

The source table contains one dividend event per symbol. The first table preserves its event-level layout; the second expands every declaration, ex-dividend, record, and payment date into a chronological timeline.

In [2]:
NASDAQ_DATE_COLUMNS = ['declaration_date', 'ex_dividend_date', 'record_date', 'payment_date']
nasdaq = read_csv_records(OUTPUT_DIR / 'nasdaq.csv')

for row in nasdaq:
    for column in NASDAQ_DATE_COLUMNS:
        row[column] = date.fromisoformat(row[column])
    row['amount'] = Decimal(row['amount'])

nasdaq.sort(key=lambda row: (row['ex_dividend_date'], row['symbol']))
show_table(
    nasdaq,
    ['symbol', 'declaration_date', 'ex_dividend_date', 'record_date', 'payment_date', 'amount'],
)

symbol,declaration_date,ex_dividend_date,record_date,payment_date,amount
AAPL,2026-07-30,2026-08-10,2026-08-10,2026-08-13,0.27
MSFT,2026-06-10,2026-08-20,2026-08-20,2026-09-10,0.91
GOOGL,2026-07-22,2026-09-04,2026-09-07,2026-09-14,0.22
WDC,2026-08-04,2026-09-08,2026-09-08,2026-09-17,0.15
NVDA,2026-08-26,2026-09-10,2026-09-10,2026-10-01,0.25


In [3]:
event_labels = {
    'declaration_date': 'declaration',
    'ex_dividend_date': 'ex-dividend',
    'record_date': 'record',
    'payment_date': 'payment',
}
nasdaq_timeline = sorted(
    (
        {
            'date': row[column],
            'symbol': row['symbol'],
            'event': event_labels[column],
            'dividend_per_share': row['amount'],
        }
        for row in nasdaq
        for column in NASDAQ_DATE_COLUMNS
    ),
    key=lambda row: (row['date'], row['symbol'], row['event']),
)
show_table(nasdaq_timeline)

date,symbol,event,dividend_per_share
2026-06-10,MSFT,declaration,0.91
2026-07-22,GOOGL,declaration,0.22
2026-07-30,AAPL,declaration,0.27
2026-08-04,WDC,declaration,0.15
2026-08-10,AAPL,ex-dividend,0.27
2026-08-10,AAPL,record,0.27
2026-08-13,AAPL,payment,0.27
2026-08-20,MSFT,ex-dividend,0.91
2026-08-20,MSFT,record,0.91
2026-08-26,NVDA,declaration,0.25


## Implied dividend valuation method

Daily unadjusted close prices come from Yahoo Finance's [chart endpoint](https://query1.finance.yahoo.com/v8/finance/chart/AAPL). For each effective date, the notebook requests a window from seven calendar days before through two days after. It uses that date's close when available; for a weekend or market holiday, it uses the most recent earlier trading close and reports the selected Yahoo session date.

For snapshot supply $S$, snapshot multiplier $M_0$, new multiplier $M_1$, and selected close price $P$:

1. Total underlying shares: $Q = S / M_0$
2. Newly issued stock equivalent: $\Delta Q = Q M_1 - Q M_0$
3. Total implied dividend value: $V = \Delta Q \times P$
4. Implied value per snapshot token: $D = V / S$
5. Difference from the Nasdaq declaration: $(D - D_{declared}) / D_{declared} \times 100\%$
6. Multiplier change: $(M_1 - M_0) / M_0 \times 100\%$

In the provider CSVs, total_supply is the supply read at total_supply_snapshot_block, previous_multiplier is $M_0$, and proposed_multiplier is $M_1$. Calculations use Decimal with 50-digit working precision; rounding is only applied to displayed results.

In [4]:
YAHOO_CHART_URL = 'https://query1.finance.yahoo.com/v8/finance/chart'
PRICE_LOOKBACK_DAYS = 7
PRICE_LOOKAHEAD_DAYS = 2
YAHOO_HEADERS = {'User-Agent': 'Mozilla/5.0'}

nasdaq_by_symbol = {row['symbol']: row for row in nasdaq}
robinhood = read_csv_records(OUTPUT_DIR / 'robinhood.csv')
xstocks = read_csv_records(OUTPUT_DIR / 'xstocks.csv')


def yahoo_daily_closes(symbol, start_date, end_date, session):
    period1 = int(datetime.combine(start_date, time.min, tzinfo=UTC).timestamp())
    period2 = int(datetime.combine(end_date + timedelta(days=1), time.min, tzinfo=UTC).timestamp())
    response = session.get(
        f'{YAHOO_CHART_URL}/{symbol}',
        params={
            'period1': period1,
            'period2': period2,
            'interval': '1d',
            'events': 'history',
        },
        headers=YAHOO_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    chart = response.json()['chart']
    if chart.get('error'):
        raise RuntimeError(f"Yahoo Finance error for {symbol}: {chart['error']}")
    if not chart.get('result'):
        raise RuntimeError(f'Yahoo Finance returned no chart data for {symbol}.')

    result = chart['result'][0]
    exchange_tz = ZoneInfo(result['meta'].get('exchangeTimezoneName', 'UTC'))
    timestamps = result.get('timestamp', [])
    closes = result.get('indicators', {}).get('quote', [{}])[0].get('close', [])
    return {
        datetime.fromtimestamp(timestamp, tz=exchange_tz).date(): Decimal(str(close))
        for timestamp, close in zip(timestamps, closes)
        if close is not None
    }


def close_on_or_before(history, target_date):
    available_dates = [session_date for session_date in history if session_date <= target_date]
    if not available_dates:
        raise RuntimeError(f'No Yahoo close on or before {target_date} in the requested range.')
    selected_date = max(available_dates)
    return selected_date, history[selected_date]


def rounded_decimal(value, places):
    return value.quantize(Decimal(1).scaleb(-places))


effective_dates_by_symbol = {}
for event in [*robinhood, *xstocks]:
    effective_dates_by_symbol.setdefault(event['symbol'], []).append(parse_utc(event['effective_at_iso']).date())

price_history_by_symbol = {}
with requests.Session() as yahoo_session:
    for symbol, effective_dates in sorted(effective_dates_by_symbol.items()):
        window_start = min(effective_dates) - timedelta(days=PRICE_LOOKBACK_DAYS)
        window_end = max(effective_dates) + timedelta(days=PRICE_LOOKAHEAD_DAYS)
        price_history_by_symbol[symbol] = yahoo_daily_closes(
            symbol, window_start, window_end, yahoo_session
        )


def calculate_dividend_valuations(events):
    valuations = []
    for event in events:
        reference = nasdaq_by_symbol.get(event['symbol'])
        if reference is None:
            continue

        effective_date = parse_utc(event['effective_at_iso']).date()
        price_date, close_price = close_on_or_before(
            price_history_by_symbol[event['symbol']], effective_date
        )
        snapshot_total_supply = Decimal(event['total_supply'])
        snapshot_multiplier = Decimal(event['previous_multiplier'])
        new_multiplier = Decimal(event['proposed_multiplier'])
        declared_per_share = reference['amount']

        with localcontext() as context:
            context.prec = 50
            total_underlying_shares = snapshot_total_supply / snapshot_multiplier
            new_stocks_issued = (
                total_underlying_shares * new_multiplier
                - total_underlying_shares * snapshot_multiplier
            )
            total_dividend_value = new_stocks_issued * close_price
            value_per_share = total_dividend_value / snapshot_total_supply
            percentage_difference = (
                (value_per_share - declared_per_share) / declared_per_share * Decimal(100)
            )
            multiplier_change_percent = (
                (new_multiplier - snapshot_multiplier) / snapshot_multiplier * Decimal(100)
            )

        valuations.append({
            'symbol': event['symbol'],
            'effective_date': effective_date,
            'yahoo_close_date': price_date,
            'close_usd': rounded_decimal(close_price, 4),
            'snapshot_total_supply': rounded_decimal(snapshot_total_supply, 9),
            'snapshot_multiplier': snapshot_multiplier,
            'new_multiplier': new_multiplier,
            'multiplier_change_percent': rounded_decimal(multiplier_change_percent, 6),
            'total_underlying_shares': rounded_decimal(total_underlying_shares, 9),
            'new_stocks_issued': rounded_decimal(new_stocks_issued, 9),
            'total_dividend_value_usd': rounded_decimal(total_dividend_value, 6),
            'value_per_share_usd': rounded_decimal(value_per_share, 6),
            'declared_per_share_usd': declared_per_share,
            'difference_percent': rounded_decimal(percentage_difference, 4),
        })

    return sorted(valuations, key=lambda row: (row['effective_date'], row['symbol']))

## Robinhood Stock Tokens

The events are split by the Nasdaq date they are closest to. Because every update after payment is also after the ex-date, using the closest reference avoids listing the same stock twice. The difference is shown in calendar days.

Robinhood's [Early Dividends documentation](https://robinhood.com/us/en/support/articles/dividends/#early-dividends) says eligible stocks can receive certain dividend payments before the issuer's announced payment date. Robinhood notes that the feature changes only when eligible funds are credited, not the dividend amount or tax timing. This provides context for Robinhood dividend timing; it does not document Stock Token multiplier policy.

In [5]:
robinhood_comparison = []

for event in robinhood:
    reference = nasdaq_by_symbol.get(event['symbol'])
    if reference is None:
        continue
    effective_at = parse_utc(event['effective_at_iso'])
    effective_date = effective_at.date()
    days_from_ex = (effective_date - reference['ex_dividend_date']).days
    days_from_payment = (effective_date - reference['payment_date']).days
    closest = 'ex-date' if abs(days_from_ex) <= abs(days_from_payment) else 'payment date'
    robinhood_comparison.append({
        'symbol': event['symbol'],
        'ex_date': reference['ex_dividend_date'],
        'payment_date': reference['payment_date'],
        'multiplier_effective_date': effective_date,
        'days_from_ex': days_from_ex,
        'days_from_payment': days_from_payment,
        'nearest_reference': closest,
    })

robinhood_comparison.sort(key=lambda row: (row['multiplier_effective_date'], row['symbol']))

In [6]:
ex_date_updates = [
    {
        'symbol': row['symbol'],
        'ex_date': row['ex_date'],
        'multiplier_effective_date': row['multiplier_effective_date'],
        'days_after_ex_date': row['days_from_ex'],
    }
    for row in robinhood_comparison
    if row['nearest_reference'] == 'ex-date'
]
payment_date_updates = [
    {
        'symbol': row['symbol'],
        'payment_date': row['payment_date'],
        'multiplier_effective_date': row['multiplier_effective_date'],
        'days_after_payment_date': row['days_from_payment'],
    }
    for row in robinhood_comparison
    if row['nearest_reference'] == 'payment date'
]

display(Markdown('### Updates aligned with the ex-date'))
show_table(ex_date_updates)
display(Markdown('### Updates after the payment date'))
show_table(payment_date_updates)

robinhood_valuations = calculate_dividend_valuations(robinhood)
display(Markdown('### Implied payout compared with the declared dividend'))
show_table(robinhood_valuations)

### Updates aligned with the ex-date

symbol,ex_date,multiplier_effective_date,days_after_ex_date
WDC,2026-09-08,2026-09-08,0
NVDA,2026-09-10,2026-09-10,0


### Updates after the payment date

symbol,payment_date,multiplier_effective_date,days_after_payment_date
AAPL,2026-08-13,2026-08-14,1
MSFT,2026-09-10,2026-09-11,1
GOOGL,2026-09-14,2026-09-15,1


### Implied payout compared with the declared dividend

symbol,effective_date,yahoo_close_date,close_usd,snapshot_total_supply,snapshot_multiplier,new_multiplier,multiplier_change_percent,total_underlying_shares,new_stocks_issued,total_dividend_value_usd,value_per_share_usd,declared_per_share_usd,difference_percent
AAPL,2026-08-14,2026-08-14,305.9300,5866.520000000,1,1.000566080061092436,0.056608,5866.520000000,3.320920000,1015.969031,0.173181,0.27,-35.8589
WDC,2026-09-08,2026-09-08,477.3000,78.735000000,1,1.000217565250523909,0.021757,78.735000000,0.017130000,8.176149,0.103844,0.15,-30.7707
NVDA,2026-09-10,2026-09-10,218.3600,85313.395000000,1,1.000775159164630595,0.077516,85313.395000000,66.131460000,14440.465646,0.169264,0.25,-32.2945
MSFT,2026-09-11,2026-09-11,495.6300,5876.607000000,1,1.000412952576205964,0.041295,5876.607000000,2.426760000,1202.775071,0.204672,0.91,-77.5086
GOOGL,2026-09-15,2026-09-15,344.9800,15777.178000000,1,1.000193924414112587,0.019392,15777.178000000,3.059580000,1055.493942,0.066900,0.22,-69.5909


## xStocks

The collected `MultiplierScheduled` event has two relevant times:

- `block_at_iso`: when the schedule was published on-chain.
- `effective_at_iso` / `effective_timestamp`: when the proposed multiplier activates.

The current [xStocks multiplier-update documentation](https://docs.xstocks.fi/docs/dividends-and-stock-splits#multiplier-updates) says the multiplier is published on-chain before the corporate event takes effect and activates at **00:30 UTC on the day immediately following the Ex-Date**. The developer guide also says consumers should retain the pending multiplier and adopt it when its activation timestamp is reached.

The next table independently checks (1) that the integer Unix timestamp, ISO timestamp, and stored effective date describe the same UTC instant, and (2) whether that instant equals the documented time derived from the Nasdaq ex-date.

In [7]:
xstocks_comparison = []

for event in xstocks:
    reference = nasdaq_by_symbol.get(event['symbol'])
    if reference is None:
        continue
    scheduled_at = parse_utc(event['block_at_iso'])
    effective_at = parse_utc(event['effective_at_iso'])
    epoch_at = datetime.fromtimestamp(int(event['effective_timestamp']), tz=UTC)
    documented_effective_at = datetime.combine(
        reference['ex_dividend_date'] + timedelta(days=1),
        time(hour=0, minute=30),
        tzinfo=UTC,
    )
    xstocks_comparison.append({
        'symbol': event['symbol'],
        'nasdaq_ex_date': reference['ex_dividend_date'],
        'scheduled_onchain_at_utc': scheduled_at,
        'effective_timestamp': event['effective_timestamp'],
        'effective_at_utc': effective_at,
        'schedule_lead_hours': round((effective_at - scheduled_at).total_seconds() / 3600, 3),
        'epoch_matches_iso': epoch_at == effective_at,
        'date_matches_iso': event['effective_date'] == effective_at.date().isoformat(),
        'docs_expected_at_utc': documented_effective_at,
        'hours_from_docs_rule': round((effective_at - documented_effective_at).total_seconds() / 3600, 3),
        'matches_docs_rule': effective_at == documented_effective_at,
    })

xstocks_comparison.sort(key=lambda row: (row['effective_at_utc'], row['symbol']))
show_table(xstocks_comparison)

symbol,nasdaq_ex_date,scheduled_onchain_at_utc,effective_timestamp,effective_at_utc,schedule_lead_hours,epoch_matches_iso,date_matches_iso,docs_expected_at_utc,hours_from_docs_rule,matches_docs_rule
AAPL,2026-08-10,2026-08-07T20:22:35Z,1786149000,2026-08-08T00:30:00Z,4.124,True,True,2026-08-11T00:30:00Z,-72.0,False
MSFT,2026-08-20,2026-08-19T20:28:35Z,1787185800,2026-08-20T00:30:00Z,4.024,True,True,2026-08-21T00:30:00Z,-24.0,False
GOOGL,2026-09-04,2026-09-03T20:36:47Z,1788481800,2026-09-04T00:30:00Z,3.887,True,True,2026-09-05T00:30:00Z,-24.0,False
NVDA,2026-09-10,2026-09-09T20:29:59Z,1789000200,2026-09-10T00:30:00Z,4.0,True,True,2026-09-11T00:30:00Z,-24.0,False


In [8]:
timestamp_checks_pass = all(
    row['epoch_matches_iso'] and row['date_matches_iso']
    for row in xstocks_comparison
)
docs_matches = sum(row['matches_docs_rule'] for row in xstocks_comparison)
missing_xstocks = sorted(set(nasdaq_by_symbol) - {row['symbol'] for row in xstocks_comparison})

display(Markdown(
    f"Internal UTC timestamp checks pass for every collected row: **{timestamp_checks_pass}**.  \
"
    f"Rows matching the documented ex-date + 1 day at 00:30 UTC rule: "
    f"**{docs_matches}/{len(xstocks_comparison)}**.  \
"
    f"Nasdaq symbols without an xStocks event: **{missing_xstocks or 'none'}**."
))

Internal UTC timestamp checks pass for every collected row: **True**.  Rows matching the documented ex-date + 1 day at 00:30 UTC rule: **0/4**.  Nasdaq symbols without an xStocks event: **['WDC']**.

A zero or nonzero `hours_from_docs_rule` is the direct rule check. A mismatch does **not** by itself show that the contract or documentation is wrong: the Nasdaq row and on-chain event may use different ex-date sources, may not refer to the same corporate action, or may come from snapshots produced under a different scheduling convention. It identifies the rows that need source-level follow-up.

In [9]:
xstocks_valuations = calculate_dividend_valuations(xstocks)
display(Markdown('### Implied payout compared with the declared dividend'))
show_table(xstocks_valuations)

### Implied payout compared with the declared dividend

symbol,effective_date,yahoo_close_date,close_usd,snapshot_total_supply,snapshot_multiplier,new_multiplier,multiplier_change_percent,total_underlying_shares,new_stocks_issued,total_dividend_value_usd,value_per_share_usd,declared_per_share_usd,difference_percent
AAPL,2026-08-08,2026-08-07,313.3300,74100.021769937,1.0026642075893797,1.0032690125398187,0.060320,73903.128494124,44.696977966,14004.903506,0.189000,0.27,-30.0000
MSFT,2026-08-20,2026-08-20,481.1500,21573.649088613,1.0045820905025638,1.0059033904787456,0.131527,21475.247560725,28.375244090,13652.748521,0.632844,0.91,-30.4567
GOOGL,2026-09-04,2026-09-04,338.4600,69072.991298786,1.001926722393864,1.0023772500603487,0.044966,68940.162743391,31.059450648,10512.381401,0.152192,0.22,-30.8217
NVDA,2026-09-10,2026-09-10,218.3600,110094.400550734,1.0009180758490996,1.001701196801074,0.078240,109993.418249879,86.138150411,18809.126576,0.170845,0.25,-31.6618


## Reality

Reality events would be compared with the same Nasdaq dividend dates used above. However, `output/reality.csv` currently contains only its column headers and **zero data rows**, so there is no Reality timeline to analyze.